
# IndiaKart E-Commerce Analytics
## Phase 1: Data Cleaning and Data Quality Report

**Prepared by:** Raghavendra

### Objective
The objective of this phase is to inspect the eight IndiaKart CSV datasets, identify missing values and duplicates, validate data types, check relationships between tables, and document data quality findings.

### Tools Used
- Python
- Pandas
- Visual Studio Code
- Jupyter Notebook


In [2]:

# Import the required library
import pandas as pd
from pathlib import Path

# Find the project folder
project_folder = Path.cwd().parent

#  Define the eight CSV file names
file_names = {
    "customers": "customers DA.csv",
    "orders": "orders DA.csv",
    "order_items": "order_items DA.csv",
    "products": "products DA.csv",
    "payments": "payments DA.csv",
    "returns": "returns DA.csv",
    "inventory": "inventory DA.csv",
    "suppliers": "suppliers DA.csv"
}

# Load all CSV files into Pandas
tables = {}

for table_name, file_name in file_names.items():
    file_path = project_folder / file_name
    tables[table_name] = pd.read_csv(file_path)

# Confirm that all eight tables loaded
print("All tables loaded successfully!\n")

for table_name, df in tables.items():
    print(f"{table_name}: {df.shape[0]} rows, {df.shape[1]} columns")


All tables loaded successfully!

customers: 10000 rows, 17 columns
orders: 50000 rows, 19 columns
order_items: 100000 rows, 11 columns
products: 1000 rows, 17 columns
payments: 50000 rows, 13 columns
returns: 10000 rows, 10 columns
inventory: 1000 rows, 11 columns
suppliers: 200 rows, 14 columns


In [3]:

# Profile all eight datasets

for table_name, df in tables.items():
    print("\n" + "=" * 60)
    print(f"TABLE: {table_name.upper()}")
    print("=" * 60)

    print("Rows and columns:", df.shape)

    print("\nColumn data types:")
    print(df.dtypes)

    print("\nMissing values per column:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())



TABLE: CUSTOMERS
Rows and columns: (10000, 17)

Column data types:
customer_id           object
first_name            object
last_name             object
email                 object
phone                  int64
city                  object
state                 object
pincode                int64
gender                object
age                    int64
segment               object
registration_date     object
last_login_date       object
is_verified            int64
is_active              int64
total_orders           int64
total_spent          float64
dtype: object

Missing values per column:
customer_id          0
first_name           0
last_name            0
email                0
phone                0
city                 0
state                0
pincode              0
gender               0
age                  0
segment              0
registration_date    0
last_login_date      0
is_verified          0
is_active            0
total_orders         0
total_spent          0
dtype:

In [4]:

# Missing Value Counts and Percentages

for table_name, df in tables.items():
    missing_count = df.isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100

    missing_report = pd.DataFrame({
        "Missing Count": missing_count,
        "Missing Percentage": missing_percentage.round(2)
    })

    missing_report = missing_report[
        missing_report["Missing Count"] > 0
    ]

    print(f"\n--- {table_name.upper()} ---")

    if missing_report.empty:
        print("No missing values found.")
    else:
        print(missing_report)



--- CUSTOMERS ---
No missing values found.

--- ORDERS ---
                Missing Count  Missing Percentage
delivered_date          17501                35.0

--- ORDER_ITEMS ---
No missing values found.

--- PRODUCTS ---
No missing values found.

--- PAYMENTS ---
             Missing Count  Missing Percentage
refund_date          50000               100.0

--- RETURNS ---
No missing values found.

--- INVENTORY ---
No missing values found.

--- SUPPLIERS ---
No missing values found.


In [5]:

# Check Duplicate Records and IDs

print("--- Duplicate Checks ---")

for table_name, df in tables.items():
    print(f"\nTable: {table_name}")
    print("Duplicate rows:", df.duplicated().sum())

# Check unique identifiers in important tables
id_checks = {
    "customers": "customer_id",
    "orders": "order_id",
    "order_items": "item_id",
    "products": "product_id",
    "payments": "payment_id",
    "returns": "return_id",
    "inventory": "inventory_id",
    "suppliers": "supplier_id"
}

for table_name, id_column in id_checks.items():
    df = tables[table_name]

    if id_column in df.columns:
        print(
            f"{table_name} - duplicate {id_column}:",
            df[id_column].duplicated().sum()
        )


--- Duplicate Checks ---

Table: customers
Duplicate rows: 0

Table: orders
Duplicate rows: 0

Table: order_items
Duplicate rows: 0

Table: products
Duplicate rows: 0

Table: payments
Duplicate rows: 0

Table: returns
Duplicate rows: 0

Table: inventory
Duplicate rows: 0

Table: suppliers
Duplicate rows: 0
customers - duplicate customer_id: 0
orders - duplicate order_id: 0
order_items - duplicate item_id: 0
products - duplicate product_id: 0
payments - duplicate payment_id: 0
returns - duplicate return_id: 0
inventory - duplicate inventory_id: 0
suppliers - duplicate supplier_id: 0


In [6]:

#  Verify Relationships Between Tables

orders = tables["orders"]
order_items = tables["order_items"]
returns = tables["returns"]

# Check 1: Order items must reference existing orders
order_ids = set(orders["order_id"])

orphan_items = order_items[
    ~order_items["order_id"].isin(order_ids)
]

print("--- Order Items Relationship Check ---")
print("Total order items:", len(order_items))
print("Unmatched order IDs:", len(orphan_items))

# Check 2: Returns must reference existing orders
orphan_returns = returns[
    ~returns["order_id"].isin(order_ids)
]

print("\n--- Returns Relationship Check ---")
print("Total returns:", len(returns))
print("Unmatched order IDs:", len(orphan_returns))


--- Order Items Relationship Check ---
Total order items: 100000
Unmatched order IDs: 0

--- Returns Relationship Check ---
Total returns: 10000
Unmatched order IDs: 0


In [7]:

#  Validate Delivery Dates

orders = tables["orders"].copy()

# Convert date columns to datetime
orders["order_date"] = pd.to_datetime(
    orders["order_date"], dayfirst=True, errors="coerce"
)

orders["delivered_date"] = pd.to_datetime(
    orders["delivered_date"], dayfirst=True, errors="coerce"
)

# Select orders marked as Delivered
delivered_orders = orders[
    orders["status"] == "Delivered"
]

# Find delivered orders with missing delivery dates
missing_delivery_dates = delivered_orders[
    delivered_orders["delivered_date"].isna()
]

# Find delivery dates earlier than order dates
invalid_dates = delivered_orders[
    delivered_orders["delivered_date"].notna()
    & (delivered_orders["delivered_date"] < delivered_orders["order_date"])
]

print("--- Delivery Date Validation ---")
print("Delivered orders:", len(delivered_orders))
print("Delivered orders with missing delivery dates:",
      len(missing_delivery_dates))
print("Delivery dates earlier than order dates:",
      len(invalid_dates))


--- Delivery Date Validation ---
Delivered orders: 32499
Delivered orders with missing delivery dates: 0
Delivery dates earlier than order dates: 0


In [8]:

#  Payment Success and Failure Analysis

payments = tables["payments"]

payment_counts = payments["status"].value_counts()

success_count = (payments["status"] == "Success").sum()
failed_count = (payments["status"] == "Failed").sum()
total_payments = len(payments)

failure_rate = failed_count / total_payments * 100

print("--- Payment Status Check ---")
print("Total payments:", total_payments)
print("\nPayment status counts:")
print(payment_counts)
print(f"\nPayment failure rate: {failure_rate:.2f}%")
print("PRD target: Below 2%")


--- Payment Status Check ---
Total payments: 50000

Payment status counts:
status
Success    48252
Failed      1748
Name: count, dtype: int64

Payment failure rate: 3.50%
PRD target: Below 2%


In [9]:

#  Inventory Stock Status Check

inventory = tables["inventory"]

print("--- Inventory Stock Status Check ---")

print("Total inventory records:", len(inventory))

print("\nStock status counts:")
print(inventory["status"].value_counts(dropna=False))

out_of_stock = inventory[
    inventory["status"].astype(str).str.strip().str.lower()
    == "out of stock"
]

print("\nTotal Out of Stock records:", len(out_of_stock))


--- Inventory Stock Status Check ---
Total inventory records: 1000

Stock status counts:
status
In Stock        922
Low Stock        76
Out of Stock      2
Name: count, dtype: int64

Total Out of Stock records: 2


In [10]:

#  Customers With Zero Orders

customers = tables["customers"]

zero_order_customers = customers[
    customers["total_orders"] == 0
]

print("--- Customers With Zero Orders ---")
print("Total customers:", len(customers))
print("Customers with zero orders:", len(zero_order_customers))

percentage = len(zero_order_customers) / len(customers) * 100

print(f"Percentage with zero orders: {percentage:.2f}%")


--- Customers With Zero Orders ---
Total customers: 10000
Customers with zero orders: 457
Percentage with zero orders: 4.57%


In [11]:

#  High-Value Order Check

orders = tables["orders"]

large_orders = orders[
    orders["final_amount"] > 500000
]

print("--- High-Value Order Check ---")
print("Threshold: ₹5,00,000")
print("Orders above threshold:", len(large_orders))

print("\nTop 10 highest-value orders:")

print(
    large_orders[
        ["order_id", "customer_id", "status", "final_amount"]
    ].sort_values(
        "final_amount", ascending=False
    ).head(10)
)


--- High-Value Order Check ---
Threshold: ₹5,00,000
Orders above threshold: 312

Top 10 highest-value orders:
        order_id customer_id     status  final_amount
27602  ORD027603   CUST07346   Returned    1133382.48
10854  ORD010855   CUST04733  Delivered     967333.45
2615   ORD002616   CUST07574  Delivered     943168.29
15469  ORD015470   CUST09672  Delivered     892302.73
19553  ORD019554   CUST04064   Returned     885092.35
8819   ORD008820   CUST04550  Delivered     878994.69
36722  ORD036723   CUST05128    Shipped     861024.42
17396  ORD017397   CUST04026  Delivered     860430.57
17487  ORD017488   CUST09740  Delivered     850556.16
21473  ORD021474   CUST07425   Returned     817893.92


In [12]:

#  Check Numeric Values

numeric_checks = {
    "orders": [
        "total_amount", "gst_amount",
        "discount_amount", "final_amount"
    ],
    "payments": ["amount", "refund_amount"],
    "order_items": ["quantity", "unit_price"],
    "inventory": [
        "quantity_available", "quantity_reserved",
        "unit_cost", "total_inventory_value"
    ]
}

print("--- Numeric Data Quality Check ---")

for table_name, columns in numeric_checks.items():
    df = tables[table_name]

    print(f"\nTable: {table_name}")

    for column in columns:
        if column in df.columns:
            negative_count = (df[column] < 0).sum()
            print(f"{column}: {negative_count} negative values")


--- Numeric Data Quality Check ---

Table: orders
total_amount: 0 negative values
gst_amount: 0 negative values
discount_amount: 0 negative values
final_amount: 0 negative values

Table: payments
amount: 0 negative values
refund_amount: 0 negative values

Table: order_items
quantity: 0 negative values
unit_price: 0 negative values

Table: inventory
quantity_available: 0 negative values
quantity_reserved: 0 negative values
unit_cost: 0 negative values
total_inventory_value: 0 negative values


In [13]:

# Phase 1 Data Quality Summary

print("=" * 55)
print("INDIAKART - PHASE 1 DATA QUALITY SUMMARY")
print("=" * 55)

print("\n1. DATASET SIZE")
total_rows = sum(len(df) for df in tables.values())
print("Tables inspected:", len(tables))
print("Total records across all tables:", total_rows)

print("\n2. DUPLICATE ORDER IDs")
print(
    "Duplicate order IDs:",
    tables["orders"]["order_id"].duplicated().sum()
)

print("\n3. MISSING VALUES")
for table_name, df in tables.items():
    missing = int(df.isnull().sum().sum())
    print(f"{table_name}: {missing} missing cells")

print("\n4. HIGH-VALUE ORDERS")
print(
    "Orders above ₹5 lakh:",
    (tables["orders"]["final_amount"] > 500000).sum()
)

print("\n5. PAYMENT FAILURES")
payments = tables["payments"]
failed = (payments["status"] == "Failed").sum()
print("Failed payments:", failed)
print(f"Failure rate: {failed / len(payments) * 100:.2f}%")

print("\n6. INVENTORY")
inventory = tables["inventory"]
out_of_stock = (
    inventory["status"].astype(str).str.strip().str.lower()
    == "out of stock"
).sum()
print("Out of Stock records:", out_of_stock)

print("\n7. CUSTOMERS WITH ZERO ORDERS")
customers = tables["customers"]
zero_orders = (customers["total_orders"] == 0).sum()
print("Customers with zero orders:", zero_orders)

print("\nReview flagged findings before changing any data.")


INDIAKART - PHASE 1 DATA QUALITY SUMMARY

1. DATASET SIZE
Tables inspected: 8
Total records across all tables: 222200

2. DUPLICATE ORDER IDs
Duplicate order IDs: 0

3. MISSING VALUES
customers: 0 missing cells
orders: 17501 missing cells
order_items: 0 missing cells
products: 0 missing cells
payments: 50000 missing cells
returns: 0 missing cells
inventory: 0 missing cells
suppliers: 0 missing cells

4. HIGH-VALUE ORDERS
Orders above ₹5 lakh: 312

5. PAYMENT FAILURES
Failed payments: 1748
Failure rate: 3.50%

6. INVENTORY
Out of Stock records: 2

7. CUSTOMERS WITH ZERO ORDERS
Customers with zero orders: 457

Review flagged findings before changing any data.


In [14]:

# Inspect Date Columns Across All Tables

date_tables = [
    "customers",
    "products",
    "payments",
    "returns",
    "inventory",
    "suppliers"
]


print("--- Date Column Validation ---")

for table_name in date_tables:
    df = tables[table_name]

    # Find columns whose names contain "date"
    date_columns = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    print(f"\nTABLE: {table_name.upper()}")

    if not date_columns:
        print("No date columns found.")
        continue

    for column in date_columns:
        original = df[column]

        # Convert a temporary copy for validation
        parsed = pd.to_datetime(
            original,
            dayfirst=True,
            errors="coerce"
        )

        # Count non-empty values that failed conversion
        non_empty = (
            original.notna()
            & original.astype(str).str.strip().ne("")
        )
        invalid_count = (non_empty & parsed.isna()).sum()

        print(f"\nColumn: {column}")
        print("Original data type:", original.dtype)
        print("Missing values:", original.isna().sum())
        print("Invalid non-empty dates:", invalid_count)
        print("Successfully parsed dates:", parsed.notna().sum())


--- Date Column Validation ---

TABLE: CUSTOMERS

Column: registration_date
Original data type: object
Missing values: 0
Invalid non-empty dates: 0
Successfully parsed dates: 10000

Column: last_login_date
Original data type: object
Missing values: 0
Invalid non-empty dates: 0
Successfully parsed dates: 10000

TABLE: PRODUCTS

Column: launch_date
Original data type: object
Missing values: 0
Invalid non-empty dates: 0
Successfully parsed dates: 1000

TABLE: PAYMENTS

Column: payment_date
Original data type: object
Missing values: 0
Invalid non-empty dates: 0
Successfully parsed dates: 50000

Column: refund_date
Original data type: float64
Missing values: 50000
Invalid non-empty dates: 0
Successfully parsed dates: 0

TABLE: RETURNS

Column: return_date
Original data type: object
Missing values: 0
Invalid non-empty dates: 0
Successfully parsed dates: 10000

Column: refund_date
Original data type: object
Missing values: 0
Invalid non-empty dates: 0
Successfully parsed dates: 10000

TABLE: 

In [15]:

#  Compact Date Validation Summary

date_tables = [
    "returns",
    "inventory",
    "suppliers"
]

for table_name in date_tables:
    df = tables[table_name]
    date_columns = [
        col for col in df.columns
        if "date" in col.lower()
    ]

    print(f"\n--- {table_name.upper()} ---")

    for col in date_columns:
        parsed = pd.to_datetime(
            df[col], dayfirst=True, errors="coerce"
        )
        non_empty = (
            df[col].notna()
            & df[col].astype(str).str.strip().ne("")
        )
        invalid = (non_empty & parsed.isna()).sum()

        print(
            f"{col}: missing={df[col].isna().sum()}, "
            f"invalid={invalid}, "
            f"parsed={parsed.notna().sum()}"
        )



--- RETURNS ---
return_date: missing=0, invalid=0, parsed=10000
refund_date: missing=0, invalid=0, parsed=10000

--- INVENTORY ---
last_restocked_date: missing=0, invalid=0, parsed=1000

--- SUPPLIERS ---
created_date: missing=0, invalid=0, parsed=200


In [16]:

#  Reconcile High-Value Orders with Order Items

orders = tables["orders"]
order_items = tables["order_items"]

# Select orders above ₹5 lakh
large_orders = orders[
    orders["final_amount"] > 500000
].copy()

# Calculate total item value for each order
item_totals = (
    order_items.groupby("order_id")["total_price"]
    .sum()
    .reset_index()
    .rename(columns={"total_price": "item_total"})
)

# Match order amounts with item totals
reconciliation = large_orders[
    ["order_id", "status", "final_amount"]
].merge(
    item_totals,
    on="order_id",
    how="left"
)

# Calculate the difference
reconciliation["difference"] = (
    reconciliation["final_amount"]
    - reconciliation["item_total"]
)

print("--- High-Value Order Reconciliation ---")
print("High-value orders:", len(reconciliation))
print("Orders without item records:",
      reconciliation["item_total"].isna().sum())

print("\nDifference summary:")
print(reconciliation["difference"].describe())

print("\n10 orders with the largest absolute differences:")
reconciliation["absolute_difference"] = (
    reconciliation["difference"].abs()
)

print(
    reconciliation.sort_values(
        "absolute_difference", ascending=False
    )[
        ["order_id", "status", "final_amount",
         "item_total", "difference"]
    ].head(10).round(2)
)


--- High-Value Order Reconciliation ---
High-value orders: 312
Orders without item records: 0

Difference summary:
count    3.120000e+02
mean    -1.119378e-12
std      4.804516e-11
min     -1.164153e-10
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.164153e-10
Name: difference, dtype: float64

10 orders with the largest absolute differences:
      order_id      status  final_amount  item_total  difference
180  ORD025036   Delivered     544956.06   544956.06         0.0
138  ORD018730  Processing     650018.67   650018.67         0.0
132  ORD017920   Delivered     645927.85   645927.85        -0.0
255  ORD032706   Delivered     595789.64   595789.64         0.0
126  ORD017157     Shipped     550304.58   550304.58        -0.0
216  ORD029277   Delivered     566876.96   566876.96        -0.0
36   ORD004631   Delivered     560156.96   560156.96        -0.0
121  ORD016252   Delivered     592140.07   592140.07        -0.0
118  ORD016065   Cancelled     627429.34 